### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key in environment vars ")

os.environ["LANGSMITH_TRACING"] = "true"

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message Based summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [ ]:
# Alternative test data
questions = [
    "What is 2+2 ?",
    "What is 10*5 ?",
    "What is 100/4 ?",
    "What is 15-7 ?",
    "What is 3*3 ?",
    "What is 4*4 ?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='03c8a6e5-ff0e-4382-bf46-0db98611a786'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee69c2ef48', 'id': 'chatcmpl-Co33mqQrix3YLiaQ2qfJYy0YjVGtm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--77843735-bd86-4fcc-918a-a50f2d37e40a-0', usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}
Messa

## token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~149 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b6555abb-79cf-484f-adca-782aa6c21184'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_db64b059be', 'id': 'chatcmpl-Co38pQM7GbvNkoeyZQplLXc20kKOx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--5d9703e4-b2ec-4cce-a5b4-7bbcac8c1b9c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_T1qZPF6uGKEEVx5WcsW8bs7e', 'type': 'tool_call'}], usage_metadata={'input_tokens': 53, 'output_tokens': 15, 

### Fraction

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~64 tokens (0.0500%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a83cc0a1-77b5-4ab3-bd76-e13156000b53'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-Co3ArOQrPc3zwg995EChBheLKW5dE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--87bc14f2-8864-4945-986c-9e54ca9eb0c3-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_n5G4wPITTn9Qbn7PcmixQe95', 'type': 'tool_call'}], usage_metadata={'input_tokens': 44, 'output_tokens': 15, 

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

In [2]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

##### Test Human Action : Approve Action

In [6]:
from langchain_core.messages import HumanMessage

approve_config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'")]},
    config=approve_config
)

#
result

{'messages': [HumanMessage(content="Send email to brijeshdhaker@gmail.com with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='f2535c80-c50d-4f0a-851a-7a741ee4cf0d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to brijeshdhaker@gmail.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nLooking at the tools provided, there's a send_email_tool. The parameters required are recipient, subject, and body. The user provided all three: recipient is the email address given, subject is 'Hello', and the body is 'How are you?'. \n\nI need to make sure all required parameters are included. The function requires recipient, subject, and body, which are all present. So I should call send_email_tool with those arguments. There's no need to use the read_email_tool here since the task is about sending, not reading. \n\nDouble-checking the parameters: recipient is cor

In [7]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
#
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=approve_config
)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to brijeshdhaker@gmail.com with the subject "Hello" and the message "How are you?" 😊 Let me know if you need anything else!


In [8]:
result

{'messages': [HumanMessage(content="Send email to brijeshdhaker@gmail.com with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='f2535c80-c50d-4f0a-851a-7a741ee4cf0d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to brijeshdhaker@gmail.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nLooking at the tools provided, there's a send_email_tool. The parameters required are recipient, subject, and body. The user provided all three: recipient is the email address given, subject is 'Hello', and the body is 'How are you?'. \n\nI need to make sure all required parameters are included. The function requires recipient, subject, and body, which are all present. So I should call send_email_tool with those arguments. There's no need to use the read_email_tool here since the task is about sending, not reading. \n\nDouble-checking the parameters: recipient is cor

##### Test Human Action : Reject Action

In [9]:
reject_config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'")]},
    config=reject_config
)

In [10]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=reject_config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email to 'brijeshdhaker@gmail.com' with subject 'Hello' was rejected by the mock system (rejection ID: 7kh4e8gm7). Would you like me to:
1. Attempt to send it again with modified content?
2. Read an email instead using the read_email_tool?
3. Try a different action?

<debug>
[SYSTEM NOTE: This is a simulated rejection from the send_email_tool mock function to demonstrate error handling]
</debug>


In [11]:
result

{'messages': [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='e47d3a37-7a54-4735-9934-5d762d732e65'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to 'brijeshdhaker@gmail.com' with the subject 'Hello' and body 'How are you?'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. The send_email_tool is the right one here because the user is asking to send an email.\n\nLooking at the parameters required for send_email_tool: recipient, subject, and body. The user provided all three: recipient is the email address given, subject is 'Hello', and body is 'How are you ?'. I need to make sure the parameters match. The recipient is a string, subject is a string, and body is a string. All required parameters are present. So I should call the send_email_tool with these argum

##### Test Human Action : Editing Action 

In [12]:
edit_config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'")]},
    config=edit_config
)

In [13]:
result

{'messages': [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='855837ad-f006-486d-bd91-4b8f648417cf'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to 'wrong@email.com' with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yes, they are. So I should call the send_email_tool with these arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': '3m0fzmejn', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_t

In [15]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "neetadhk@gmail.com",
                                "subject": "Editated Subject",
                                "body": "This was edited by human before sending."
                            }
                        }
                    }
                ]
            }
        ),
        config=edit_config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email has been sent to **neetadhk@gmail.com** with the subject **"Editated Subject"** and the body **"This was edited by human before sending."** Let me know if you'd like to review or adjust anything else!


In [ ]:
result

{'messages': [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='855837ad-f006-486d-bd91-4b8f648417cf'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to 'wrong@email.com' with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yes, they are. So I should call the send_email_tool with these arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': '3m0fzmejn', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_t

: 